In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run(["random", "random"])

from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, COMET_SPAWN_STEPS

obs = env.steps[1][0].observation
planets = [Planet(*p) for p in obs.planets]

print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")

print(f"Initial Planet {obs.initial_planets}")

obs = env.steps[COMET_SPAWN_STEPS[0]][0].observation
comet_planet_ids = obs.get("comet_planet_ids", [])
print(f"Commets {len(comet_planet_ids)}")

In [100]:
# %%writefile submission.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, COMET_SPAWN_STEPS
from itertools import count

INTERCEPT_THRESHOLD = 0.016

sun_config = (50.0, 50.0, 10.0)
time = 0

fleet_target_map = {} # maps already targeted planets 
launch_history = []
my_planets_trangectory = {}

def find_planet(planets, id):
    for p in planets:
        if p.id == id:
            return p
    return None
        
def cal_intercept(planet, angular_v, launcher_pos):
    dx = planet.x - sun_config[0]
    dy = planet.y - sun_config[1]
    
    current_angle = math.atan2(dy, dx)
    planet_r = math.sqrt(dx**2 + dy**2) # distance fromt the sun's center to the target planet
    t = 1
    
    ships_needed = planet.ships + 1
    
    for _ in range(20):
        target_angle = current_angle + (angular_v * t) # calculate the next angle given the time, t
        # cal position from the new angle
        tx = sun_config[0] + (planet_r * math.cos(target_angle))
        ty = sun_config[1] + (planet_r * math.sin(target_angle))
        
        # cal the distance and time the planet would travel
        dist = math.sqrt((tx - launcher_pos.x)**2 + (ty - launcher_pos.y)**2)
        
        if planet.owner > -1:
            ships_needed = (t * planet.production) + planet.ships + 1
            
        travel_time = dist / cal_fleet_speed(ships_needed)
        
        if abs(t - travel_time) < INTERCEPT_THRESHOLD: # Threshold for 'close enough'
            break
        t = travel_time
    
    return tx, ty, ships_needed

def cal_intercept_with_sun(start_x, start_y, nearest_x, nearest_y, angle):
    dx = sun_config[0] - start_x
    dy = sun_config[1] - start_y
    
    nx = sun_config[0] - nearest_x
    ny = sun_config[1] - nearest_y
    
    miss_distance = abs(dx * math.sin(angle) - dy * math.cos(angle))
    dot_product = dx * math.cos(angle) + dy * math.sin(angle)
    
    nearest_angle = math.atan2(ny, nx)
    nearest_dot_product = nx * math.cos(nearest_angle) + ny * math.sin(nearest_angle)
    return miss_distance, dot_product, nearest_dot_product

def is_orbiting(planet):
    return cal_hypotenus(sun_config[0], sun_config[1], planet.x, planet.y) < 50

def get_angle(dx, dy, target, angular_velocity, my_planet):
    angle = math.atan2(dy, dx)
    tx, ty, ships_needed = target.x, target.y, target.ships + 1
    if is_orbiting(target) and target.owner > -1:
        # production, distance, current number of ships
        distance = cal_hypotenus(my_planet.x, my_planet.y, target.x, target.y)
        for t in range(1, 20):
            num_ships = (t * target.production) + target.ships
            speed = cal_fleet_speed(num_ships)
            travel_time = distance / speed
            
            if travel_time <= t:
                ships_needed = num_ships
                break
    elif is_orbiting(target):
        tx, ty, ships_needed = cal_intercept(target, angular_velocity, my_planet)
        angle = math.atan2(ty - my_planet.y, tx - my_planet.x)
    return angle, tx, ty, ships_needed

def get_inputs(obs):
    player = obs.get("player", 0)
    planets = [Planet(*p) for p in obs.get("planets", [])]
    initial_planets = [Planet(*p) for p in obs.get("initial_planets", [])]
    fleets = [Fleet(*f) for f in obs.get("fleets", [])]
    
    comet_planet_ids = obs.get("comet_planet_ids", [])

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    
    fleets_owned = [f for f in fleets if f.owner == player]
    other_fleets = [f for f in fleets if f.owner != player]
    
    return my_planets, targets, fleets_owned, comet_planet_ids, other_fleets

def cal_fleet_speed(num_of_ships):
    return 1.0 + (6.0 - 1.0) * (math.log(num_of_ships) / math.log(1000)) ** 1.5

def sync_new_fleets(fleet_owned):
    for f in fleet_owned:
        if f.id not in fleet_target_map:
            match = None
            for i, record in enumerate(launch_history):
                origin_id, target_id, intent_angle, turn = record
                
                if f.from_planet_id == origin_id and math.isclose(f.angle, intent_angle, abs_tol=1e-4):
                    match = target_id
                    launch_history.pop(i)
                    break
                
            if match is not None:
                fleet_target_map[f.id] = match
    
    launch_history[:] = [r for r in launch_history if time - r[3] < 3]

def cal_hypotenus(x0, y0, x1, y1):
    return math.hypot(x1 - x0, y1 - y0)

def get_moves(target, reserved_targets, mine, angular_velocity, comet_planet_ids):
    # reserved_targets - target with launched fleet
    if target.id in comet_planet_ids or target.id in reserved_targets:
        return None

    dx = target.x - mine.x
    dy = target.y - mine.y
        
    angle, tx, ty, ships_needed = get_angle(dx, dy, target, angular_velocity, mine)
        
    if mine.ships >= ships_needed:
        distance_r, dot_product, nearest_dot_product = cal_intercept_with_sun(mine.x, mine.y, tx, ty, angle)

        move = [mine.id, angle, ships_needed]
        if not (dot_product > 0 and distance_r <= sun_config[2]) or nearest_dot_product < 0:
            launch_history.append([mine.id, target.id, angle, time])
            reserved_targets.add(target.id)
            return move

def cal_my_planets_tragectories(planet, angular_v):
    if is_orbiting(planet):
        dx = planet.x - sun_config[0]
        dy = planet.y - sun_config[1]
        
        current_angle = math.atan2(dy, dx)
        planet_r = math.sqrt(dx**2 + dy**2)
        
        for time in range(1, 4):
            next_angle = current_angle + (angular_v * time)
            nx = sun_config[0] + (planet_r * math.cos(next_angle))
            ny = sun_config[1] + (planet_r * math.cos(next_angle))
            
            if planet.id in my_planets_trangectory:
                my_planets_trangectory[planet.id].append([nx, ny, time])
                
                if len(my_planets_trangectory[planet.id][0]) > 3:
                    del my_planets_trangectory[planet.id][0][0]
            else:
                my_planets_trangectory[planet.id] = [[nx, ny, time]]
    else:
        my_planets_trangectory[planet.id] = [[planet.x, planet.y, 0]]

def cal_attack_vector(fleet):
    for pid, trag in my_planets_trangectory.items():
        for coord in trag:
            distance = cal_hypotenus(fleet.x, fleet.y, coord[0], coord[1])
            travel_time = distance / cal_fleet_speed(fleet.ships)
            
            # print(f"Ship Travel Time = {travel_time}, {coord[2]}")
            if abs(travel_time - coord[2] + time) < INTERCEPT_THRESHOLD:
                print("SOS SOS SOS")
                return pid
    return None

def agent(obs):
    global time
    global fleet_target_map
    global my_planets_trangectory
    time += 1
    
    moves = []
    my_planets, targets, fleets_owned, comet_planet_ids, other_fleets = get_inputs(obs)

    if not targets:
        return []
    
    current_fleet_ids = {f.id for f in fleets_owned}
    fleet_target_map = {fid: tid for fid, tid in fleet_target_map.items() if fid in current_fleet_ids}
    my_planets_trangectory = {pid: trag for pid, trag in my_planets_trangectory.items() if pid in [mine.id for mine in my_planets]}

    sync_new_fleets(fleets_owned)
    reserved_targets = set(fleet_target_map.values())
    
    # for f in other_fleets:
    #     p = next((p for p in my_planets if cal_attack_vector(f) == p.id), None)
    #     if p != None: targets.append(p)

    if len(my_planets) > len(targets):
        for target in targets:
            # get sorted list of nearest planets owned to launch from
            next_nearest_list = sorted(my_planets, key=lambda mine: cal_hypotenus(mine.x, mine.y, target.x, target.y))
            for mine in next_nearest_list:
                # cal_my_planets_tragectories(mine, obs.angular_velocity)
                if target.id != mine.id:
                    move = get_moves(target, reserved_targets, mine, obs.angular_velocity, comet_planet_ids)
                    if move != None:
                        moves.append(move)
    else:
        for mine in my_planets:
            # cal_my_planets_tragectories(mine, obs.angular_velocity)
            # Find nearest 3 planets we don't own
            next_nearest_list = sorted(targets, key=lambda t: cal_hypotenus(mine.x, mine.y, t.x, t.y))[:3]
            for target in next_nearest_list:
                if target.id != mine.id:
                    move = get_moves(target, reserved_targets, mine, obs.angular_velocity, comet_planet_ids)
                    if move != None:
                        moves.append(move)
    return moves

In [109]:
from kaggle_environments import make

# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([agent, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env.render(mode="ipython", width=800, height=600)
with open("replay.html", "w", encoding="utf-8") as f:
    f.write(env.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE


In [55]:
from kaggle_environments import make

env4 = make("orbit_wars", debug=True)
env4.run([agent, "random", "random", "random"])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env4.render(mode="ipython", width=800, height=600)
with open("replay-multi.html", "w", encoding="utf-8") as f:
    f.write(env4.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE
Player 2: reward=-1, status=DONE
Player 3: reward=-1, status=DONE
